# Notebook 07 — Assessment Priority Engine

## Purpose
Combine:
- Exposure (Notebook 05)
- Documented difficulty (Notebook 06)
- Persistence across 2023–2025
- Official design weight (exam guidelines)

into defensible **priority bands**.

## Rules
1. Weights locked before seeing rankings
2. Difficulty normalised by exposure
3. Report bands, not only ranks
4. Run sensitivity analysis
5. Difficulty v1 = diagnostic error pressure, not national mean %

In [1]:
from pathlib import Path
import pandas as pd
import numpy as np
import json
from datetime import datetime, timezone

PROJECT_ROOT = Path.cwd().parent
PROCESSED = PROJECT_ROOT / "data" / "processed"
ANALYSIS = PROCESSED / "analysis"
DIAG = PROCESSED / "diagnostics"
MAPPED = PROCESSED / "mapped"
OUT = PROCESSED / "priority"
OUT.mkdir(parents=True, exist_ok=True)

pd.set_option("display.max_columns", None)
pd.set_option("display.width", 160)

print("OUT:", OUT)

OUT: c:\Users\Administrator\Desktop\Matric-Maths-Exam-Intelligence\data\processed\priority


In [2]:
# N05 exposure / pressure table
pressure_path = ANALYSIS / "assessment_pressure_2023_2025.csv"
freq_path = ANALYSIS / "topic_frequency_2023_2025.csv"
marks_path = ANALYSIS / "topic_mark_allocation_2023_2025.csv"

# N06
errors_path = DIAG / "diagnostic_errors_v1.csv"
error_pressure_path = DIAG / "diagnostic_error_pressure_v1.csv"

exposure = pd.read_csv(pressure_path) if pressure_path.exists() else pd.read_csv(freq_path)
errors = pd.read_csv(errors_path)
error_pressure = pd.read_csv(error_pressure_path) if error_pressure_path.exists() else None

print("Exposure columns:", exposure.columns.tolist())
print("Errors rows:", len(errors))
print("Error pressure rows:", 0 if error_pressure is None else len(error_pressure))
display(exposure.head())
display(errors.head(3))

Exposure columns: ['topic', 'subquestion_count', 'frequency_share_pct', 'total_marks', 'mark_share_pct', 'mean_marks', 'count_minus_mark_pct', 'assessment_pressure_score']
Errors rows: 273
Error pressure rows: 11


,topic,subquestion_count,frequency_share_pct,total_marks,mark_share_pct,mean_marks,count_minus_mark_pct,assessment_pressure_score
0,Calculus,45,17.24,248.0,26.47,7.515152,-9.23,21.85
1,Trigonometry,38,14.56,162.0,17.29,7.714286,-2.73,15.92
2,Functions & Graphs,39,14.94,130.0,13.87,5.000000,1.07,14.40
3,Analytical Geometry,33,12.64,114.0,12.17,4.956522,0.47,12.40
4,Algebra & Equations,23,8.81,74.0,7.90,5.285714,0.91,8.36


,year,question_number,question_title,error_text_raw,error_text_summary,source,qualitative_severity,topic
0,2023,1.0,ALGEBRA,(a) In Q1.1.2 some candidates did not write th...,NaN,DBE diagnostic,moderate_language,Algebra & Equations
1,2023,1.0,ALGEBRA,(b) In Q1.1.3 many candidates were able to squ...,NaN,DBE diagnostic,moderate_language,Algebra & Equations
2,2023,1.0,ALGEBRA,(c) Many candidates struggled to solve the ine...,NaN,DBE diagnostic,severe_language,Algebra & Equations


In [3]:
WEIGHTS = {
    "exposure": 0.30,
    "difficulty": 0.40,
    "persistence": 0.20,
    "design_weight": 0.10,
}

assert abs(sum(WEIGHTS.values()) - 1.0) < 1e-9

(OUT / "priority_weights.json").write_text(json.dumps({
    "weights": WEIGHTS,
    "locked_at": datetime.now(timezone.utc).isoformat(),
    "notes": "Locked before ranking. Difficulty uses N06 error pressure, not national mean %."
}, indent=2), encoding="utf-8")

print("WEIGHTS LOCKED:", WEIGHTS)

WEIGHTS LOCKED: {'exposure': 0.3, 'difficulty': 0.4, 'persistence': 0.2, 'design_weight': 0.1}


In [4]:
# From DBE Mathematics Examination Guidelines (approx mark allocation)
# Combined across P1+P2 for whole-subject view
DESIGN_MARKS = {
    "Algebra & Equations": 25,
    "Number Patterns & Sequences": 25,
    "Functions & Graphs": 35,
    "Finance": 15,
    "Calculus": 35,
    "Probability": 15,
    "Statistics": 20,
    "Analytical Geometry": 40,
    "Trigonometry": 50,
    "Euclidean Geometry": 40,
}

design = pd.DataFrame([
    {"topic": k, "design_marks": v} for k, v in DESIGN_MARKS.items()
])
dmin, dmax = design["design_marks"].min(), design["design_marks"].max()
design["design_norm"] = (design["design_marks"] - dmin) / (dmax - dmin)
display(design)

,topic,design_marks,design_norm
0,Algebra & Equations,25,0.285714
1,Number Patterns & Sequences,25,0.285714
2,Functions & Graphs,35,0.571429
3,Finance,15,0.000000
4,Calculus,35,0.571429
5,Probability,15,0.000000
6,Statistics,20,0.142857
7,Analytical Geometry,40,0.714286
8,Trigonometry,50,1.000000
9,Euclidean Geometry,40,0.714286


In [5]:
# --- Exposure ---
exp = exposure.copy()
# harmonise column names
if "assessment_pressure_score" in exp.columns:
    exp["exposure_score"] = exp["assessment_pressure_score"]
elif "exposure_score" not in exp.columns and "frequency_share_pct" in exp.columns:
    exp["exposure_score"] = exp["frequency_share_pct"].fillna(0)

if "subquestion_count" not in exp.columns and "subquestion_count" in errors.columns:
    pass

emin, emax = exp["exposure_score"].min(), exp["exposure_score"].max()
exp["exposure_norm"] = (exp["exposure_score"] - emin) / (emax - emin) if emax > emin else 0.0

# --- Difficulty from N06 error_pressure table if available ---
if error_pressure is not None:
    diff = error_pressure.copy()
else:
    # fallback from errors_df
    diff = (
        errors.groupby("topic")
        .agg(
            error_count=("error_text_raw", "count"),
            severe_count=("qualitative_severity", lambda s: (s == "severe_language").sum() if "qualitative_severity" in errors.columns else 0),
            years=("year", "nunique"),
        )
        .reset_index()
    )

# ensure columns
if "error_count" not in diff.columns and "documented_error_count" in diff.columns:
    diff["error_count"] = diff["documented_error_count"]
if "severe_count" not in diff.columns:
    diff["severe_count"] = 0

# join exposure counts for normalisation
if "subquestion_count" in exp.columns:
    diff = diff.merge(exp[["topic", "subquestion_count"]], on="topic", how="left")
else:
    # approximate from frequency table if needed
    if "subquestion_count" not in diff.columns:
        freq = pd.read_csv(freq_path)
        diff = diff.merge(freq[["topic", "subquestion_count"]], on="topic", how="left")

diff["subquestion_count"] = diff["subquestion_count"].fillna(1)
diff["error_pressure_raw"] = diff["error_count"] + 1.5 * diff["severe_count"]
diff["error_pressure_per_exposure"] = diff["error_pressure_raw"] / diff["subquestion_count"]

dmin, dmax = diff["error_pressure_per_exposure"].min(), diff["error_pressure_per_exposure"].max()
diff["difficulty_norm"] = (
    (diff["error_pressure_per_exposure"] - dmin) / (dmax - dmin) if dmax > dmin else 0.0
)

# --- Persistence ---
if "years" in diff.columns:
    max_years = max(int(diff["years"].max()), 1)
    diff["persistence_norm"] = diff["years"] / max_years
else:
    pers = errors.groupby("topic")["year"].nunique().reset_index(name="years_with_errors")
    max_years = max(int(errors["year"].nunique()), 1)
    pers["persistence_norm"] = pers["years_with_errors"] / max_years
    diff = diff.merge(pers, on="topic", how="left")
    diff["persistence_norm"] = diff["persistence_norm"].fillna(0)
    max_years = max_years

print("Max years:", max_years)
display(diff[["topic", "error_count", "severe_count", "error_pressure_per_exposure", "difficulty_norm", "persistence_norm"]])

Max years: 3


,topic,error_count,severe_count,error_pressure_per_exposure,difficulty_norm,persistence_norm
0,Trigonometry,50,27,2.381579,0.340773,1.000000
1,Analytical Geometry,40,17,1.984848,0.268495,1.000000
2,Functions & Graphs,39,16,1.615385,0.201183,1.000000
3,Statistics,31,16,1.964286,0.264748,1.000000
4,Euclidean Geometry,32,11,2.852941,0.426649,1.000000
5,Algebra & Equations,15,8,1.173913,0.120753,1.000000
6,Probability,13,7,1.468750,0.174469,1.000000
7,Number Patterns & Sequences,20,6,1.705882,0.217671,1.000000
8,Finance,13,5,4.100000,0.653846,1.000000
9,Calculus,17,4,0.511111,0.000000,1.000000


In [6]:
priority = (
    exp[["topic", "exposure_norm"] + [c for c in ["subquestion_count", "total_marks", "exposure_score"] if c in exp.columns]]
    .merge(diff[["topic", "error_pressure_per_exposure", "difficulty_norm", "persistence_norm"]], on="topic", how="outer")
    .merge(design[["topic", "design_norm", "design_marks"]], on="topic", how="left")
)

priority = priority.fillna({
    "exposure_norm": 0,
    "difficulty_norm": 0,
    "persistence_norm": 0,
    "design_norm": 0.5,
    "error_pressure_per_exposure": 0,
})

priority["priority_score"] = (
    WEIGHTS["exposure"] * priority["exposure_norm"]
    + WEIGHTS["difficulty"] * priority["difficulty_norm"]
    + WEIGHTS["persistence"] * priority["persistence_norm"]
    + WEIGHTS["design_weight"] * priority["design_norm"]
).round(4)

def band(score):
    if score >= 0.65:
        return "High"
    if score >= 0.40:
        return "Medium"
    return "Lower"

priority["priority_band"] = priority["priority_score"].apply(band)
priority = priority.sort_values("priority_score", ascending=False).reset_index(drop=True)

display(priority[[
    "topic", "exposure_norm", "difficulty_norm", "persistence_norm",
    "design_norm", "priority_score", "priority_band"
]])

priority.to_csv(OUT / "priority_scores_v1.csv", index=False)
priority.to_csv(OUT / "priority_bands_v1.csv", index=False)
print("Saved priority tables")

,topic,exposure_norm,difficulty_norm,persistence_norm,design_norm,priority_score,priority_band
0,Trigonometry,0.701560,0.340773,1.000000,1.000000,0.6468,Medium
1,Calculus,1.000000,0.000000,1.000000,0.571429,0.5571,Medium
2,Analytical Geometry,0.524409,0.268495,1.000000,0.714286,0.5361,Medium
3,Functions & Graphs,0.625063,0.201183,1.000000,0.571429,0.5251,Medium
4,Unmapped,0.000000,1.000000,0.333333,0.500000,0.5167,Medium
5,Euclidean Geometry,0.244087,0.426649,1.000000,0.714286,0.5153,Medium
6,Finance,0.000000,0.653846,1.000000,0.000000,0.4615,Medium
7,Statistics,0.307499,0.264748,1.000000,0.142857,0.4124,Medium
8,Number Patterns & Sequences,0.233518,0.217671,1.000000,0.285714,0.3857,Lower
9,Algebra & Equations,0.321087,0.120753,1.000000,0.285714,0.3732,Lower


Saved priority tables


In [7]:
base_top5 = priority.head(5)["topic"].tolist()
rows = []

for dim in WEIGHTS:
    for delta in [-0.20, 0.20]:
        shifted = WEIGHTS.copy()
        shifted[dim] = max(0.0, shifted[dim] + delta)
        total = sum(shifted.values())
        shifted = {k: v / total for k, v in shifted.items()}

        score = (
            shifted["exposure"] * priority["exposure_norm"]
            + shifted["difficulty"] * priority["difficulty_norm"]
            + shifted["persistence"] * priority["persistence_norm"]
            + shifted["design_weight"] * priority["design_norm"]
        )
        top5 = priority.assign(_s=score).nlargest(5, "_s")["topic"].tolist()
        rows.append({
            "dimension_shifted": dim,
            "delta": delta,
            "top5_overlap": len(set(top5) & set(base_top5)),
            "top5_shifted": "; ".join(top5),
        })

sens = pd.DataFrame(rows)
display(sens)
sens.to_csv(OUT / "priority_sensitivity_v1.csv", index=False)
print("Min top5 overlap:", sens["top5_overlap"].min(), "/5")

,dimension_shifted,delta,top5_overlap,top5_shifted
0,exposure,-0.2,3,Unmapped; Trigonometry; Euclidean Geometry; Fi...
1,exposure,0.2,4,Trigonometry; Calculus; Functions & Graphs; An...
2,difficulty,-0.2,4,Trigonometry; Calculus; Functions & Graphs; An...
3,difficulty,0.2,3,Unmapped; Trigonometry; Euclidean Geometry; Fi...
4,persistence,-0.2,5,Unmapped; Trigonometry; Calculus; Analytical G...
5,persistence,0.2,4,Trigonometry; Calculus; Analytical Geometry; F...
6,design_weight,-0.2,5,Trigonometry; Calculus; Functions & Graphs; Un...
7,design_weight,0.2,4,Trigonometry; Analytical Geometry; Calculus; E...


Min top5 overlap: 3 /5


In [8]:
# Drop Unmapped from final priority deliverable
priority_final = priority[priority["topic"] != "Unmapped"].copy()
priority_final = priority_final.sort_values("priority_score", ascending=False).reset_index(drop=True)

display(priority_final[[
    "topic", "exposure_norm", "difficulty_norm", "persistence_norm",
    "design_norm", "priority_score", "priority_band"
]])

priority_final.to_csv(OUT / "priority_bands_final_v1.csv", index=False)

,topic,exposure_norm,difficulty_norm,persistence_norm,design_norm,priority_score,priority_band
0,Trigonometry,0.701560,0.340773,1.0,1.000000,0.6468,Medium
1,Calculus,1.000000,0.000000,1.0,0.571429,0.5571,Medium
2,Analytical Geometry,0.524409,0.268495,1.0,0.714286,0.5361,Medium
3,Functions & Graphs,0.625063,0.201183,1.0,0.571429,0.5251,Medium
4,Euclidean Geometry,0.244087,0.426649,1.0,0.714286,0.5153,Medium
5,Finance,0.000000,0.653846,1.0,0.000000,0.4615,Medium
6,Statistics,0.307499,0.264748,1.0,0.142857,0.4124,Medium
7,Number Patterns & Sequences,0.233518,0.217671,1.0,0.285714,0.3857,Lower
8,Algebra & Equations,0.321087,0.120753,1.0,0.285714,0.3732,Lower
9,Probability,0.078510,0.174469,1.0,0.000000,0.2933,Lower


In [9]:
priority_final = priority[priority["topic"] != "Unmapped"].copy()
priority_final = priority_final.sort_values("priority_score", ascending=False).reset_index(drop=True)

display(priority_final[[
    "topic", "exposure_norm", "difficulty_norm", "persistence_norm",
    "design_norm", "priority_score", "priority_band"
]])

priority_final.to_csv(OUT / "priority_bands_final_v1.csv", index=False)
print("Saved: priority_bands_final_v1.csv")

,topic,exposure_norm,difficulty_norm,persistence_norm,design_norm,priority_score,priority_band
0,Trigonometry,0.701560,0.340773,1.0,1.000000,0.6468,Medium
1,Calculus,1.000000,0.000000,1.0,0.571429,0.5571,Medium
2,Analytical Geometry,0.524409,0.268495,1.0,0.714286,0.5361,Medium
3,Functions & Graphs,0.625063,0.201183,1.0,0.571429,0.5251,Medium
4,Euclidean Geometry,0.244087,0.426649,1.0,0.714286,0.5153,Medium
5,Finance,0.000000,0.653846,1.0,0.000000,0.4615,Medium
6,Statistics,0.307499,0.264748,1.0,0.142857,0.4124,Medium
7,Number Patterns & Sequences,0.233518,0.217671,1.0,0.285714,0.3857,Lower
8,Algebra & Equations,0.321087,0.120753,1.0,0.285714,0.3732,Lower
9,Probability,0.078510,0.174469,1.0,0.000000,0.2933,Lower


Saved: priority_bands_final_v1.csv


In [10]:
DESIGN_P1 = {
    "Algebra & Equations": 25,
    "Number Patterns & Sequences": 25,
    "Functions & Graphs": 35,
    "Finance": 15,
    "Calculus": 35,
    "Probability": 15,
}
DESIGN_P2 = {
    "Statistics": 20,
    "Analytical Geometry": 40,
    "Trigonometry": 50,
    "Euclidean Geometry": 40,
}

def design_frame(d):
    df = pd.DataFrame([{"topic": k, "design_marks": v} for k, v in d.items()])
    mn, mx = df["design_marks"].min(), df["design_marks"].max()
    df["design_norm"] = (df["design_marks"] - mn) / (mx - mn) if mx > mn else 0.5
    return df

design_p1 = design_frame(DESIGN_P1)
design_p2 = design_frame(DESIGN_P2)

In [11]:
def score_paper(topic_list, design_df, label):
    sub = priority_final[priority_final["topic"].isin(topic_list)].copy()
    sub = sub.drop(columns=["design_norm", "design_marks"], errors="ignore")
    sub = sub.merge(design_df, on="topic", how="left")
    sub["design_norm"] = sub["design_norm"].fillna(0.5)

    sub["priority_score"] = (
        WEIGHTS["exposure"] * sub["exposure_norm"]
        + WEIGHTS["difficulty"] * sub["difficulty_norm"]
        + WEIGHTS["persistence"] * sub["persistence_norm"]
        + WEIGHTS["design_weight"] * sub["design_norm"]
    ).round(4)

    sub["priority_band"] = sub["priority_score"].apply(band)
    sub = sub.sort_values("priority_score", ascending=False).reset_index(drop=True)
    sub["paper"] = label
    return sub

p1_topics = list(DESIGN_P1.keys())
p2_topics = list(DESIGN_P2.keys())

priority_p1 = score_paper(p1_topics, design_p1, "P1")
priority_p2 = score_paper(p2_topics, design_p2, "P2")

print("=== PAPER 1 PRIORITY ===")
display(priority_p1[["topic", "exposure_norm", "difficulty_norm", "persistence_norm", "design_norm", "priority_score", "priority_band"]])

print("=== PAPER 2 PRIORITY ===")
display(priority_p2[["topic", "exposure_norm", "difficulty_norm", "persistence_norm", "design_norm", "priority_score", "priority_band"]])

priority_p1.to_csv(OUT / "priority_p1_v1.csv", index=False)
priority_p2.to_csv(OUT / "priority_p2_v1.csv", index=False)

=== PAPER 1 PRIORITY ===


,topic,exposure_norm,difficulty_norm,persistence_norm,design_norm,priority_score,priority_band
0,Calculus,1.000000,0.000000,1.0,1.0,0.6000,Medium
1,Functions & Graphs,0.625063,0.201183,1.0,1.0,0.5680,Medium
2,Finance,0.000000,0.653846,1.0,0.0,0.4615,Medium
3,Number Patterns & Sequences,0.233518,0.217671,1.0,0.5,0.4071,Medium
4,Algebra & Equations,0.321087,0.120753,1.0,0.5,0.3946,Lower
5,Probability,0.078510,0.174469,1.0,0.0,0.2933,Lower


=== PAPER 2 PRIORITY ===


,topic,exposure_norm,difficulty_norm,persistence_norm,design_norm,priority_score,priority_band
0,Trigonometry,0.701560,0.340773,1.0,1.000000,0.6468,Medium
1,Analytical Geometry,0.524409,0.268495,1.0,0.666667,0.5314,Medium
2,Euclidean Geometry,0.244087,0.426649,1.0,0.666667,0.5106,Medium
3,Statistics,0.307499,0.264748,1.0,0.000000,0.3981,Lower


In [12]:
def explain_row(row, paper_label="Combined"):
    return (
        f"### {row['topic']} — {row['priority_band']} ({paper_label})\n"
        f"Score: **{row['priority_score']:.3f}**\n\n"
        f"- Exposure norm: {row['exposure_norm']:.2f}\n"
        f"- Difficulty norm (error pressure / exposure): {row['difficulty_norm']:.2f}\n"
        f"- Persistence norm (years with documented errors): {row['persistence_norm']:.2f}\n"
        f"- Design weight norm: {row['design_norm']:.2f}\n"
    )

parts = [
    "# Notebook 07 — Priority Explanations\n",
    f"Weights locked: `{WEIGHTS}`\n",
    "Bands: High ≥ 0.65, Medium 0.40–0.65, Lower < 0.40\n",
    "Difficulty v1 uses DBE diagnostic error commentary, not national mean percentages.\n",
    "Unmapped topics excluded from final tables.\n",
    "\n## Combined view\n",
]
parts += [explain_row(r) for _, r in priority_final.iterrows()]
parts += ["\n## Paper 1\n"]
parts += [explain_row(r, "P1") for _, r in priority_p1.iterrows()]
parts += ["\n## Paper 2\n"]
parts += [explain_row(r, "P2") for _, r in priority_p2.iterrows()]

(OUT / "priority_explanations_v1.md").write_text("\n".join(parts), encoding="utf-8")
print("Saved: priority_explanations_v1.md")

Saved: priority_explanations_v1.md


In [13]:
summary = f"""# Notebook 07 Summary

## Status
Complete (v1)

## Weights (locked)
{json.dumps(WEIGHTS, indent=2)}

## Combined priority bands
{priority_final[['topic','priority_score','priority_band']].to_string(index=False)}

## Paper 1 top cluster
{priority_p1[['topic','priority_score','priority_band']].to_string(index=False)}

## Paper 2 top cluster
{priority_p2[['topic','priority_score','priority_band']].to_string(index=False)}

## Sensitivity
Min top-5 overlap under ±20% weight shift: {int(sens['top5_overlap'].min())}/5
Interpretation: borderline stable → report clusters, not rigid ranks.

## Limitations
- Difficulty from documented DBE error commentary (not national mean %)
- Window: 2023–2025 only
- Persistence = within this window, not full 2014–2025 history
- Unmapped excluded from final deliverables

## Core insight
Trigonometry leads the priority cluster through high design weight + documented error pressure.
Calculus remains high mainly via exposure/design weight, with lower severe-language difficulty in the current extract.
Analytical Geometry, Functions & Graphs and Euclidean Geometry form a strong secondary cluster.
"""
(OUT / "notebook07_summary.md").write_text(summary, encoding="utf-8")
print(summary)
print("\nNOTEBOOK 07 COMPLETE")

# Notebook 07 Summary

## Status
Complete (v1)

## Weights (locked)
{
  "exposure": 0.3,
  "difficulty": 0.4,
  "persistence": 0.2,
  "design_weight": 0.1
}

## Combined priority bands
                      topic  priority_score priority_band
               Trigonometry          0.6468        Medium
                   Calculus          0.5571        Medium
        Analytical Geometry          0.5361        Medium
         Functions & Graphs          0.5251        Medium
         Euclidean Geometry          0.5153        Medium
                    Finance          0.4615        Medium
                 Statistics          0.4124        Medium
Number Patterns & Sequences          0.3857         Lower
        Algebra & Equations          0.3732         Lower
                Probability          0.2933         Lower

## Paper 1 top cluster
                      topic  priority_score priority_band
                   Calculus          0.6000        Medium
         Functions & Graphs          